# Lab 9 — MLlib: pipelines e modelos

> Extraído de `aulas/Aula_7_mllib.ipynb` (seções 1 a 6 e exercícios; ajuste e deploy estão no Lab 10). As referências a números de seção usam a numeração do notebook original.

# Aula 7 — Machine Learning distribuído com MLlib
### Do DataFrame ao modelo em produção

Fechamos o ciclo: com dados limpos (aula 5), muitas vezes chegando em tempo real (aula 6), queremos **prever** algo. O MLlib é a biblioteca de machine learning do Spark. Ele treina modelos sobre DataFrames distribuídos, com a mesma lógica de pipelines do scikit-learn.

**Problema da aula:** prever quais clientes vão cancelar (*churn*) para agir antes que isso aconteça.

**Objetivos**
- Entender Transformer, Estimator e Pipeline
- Preparar features: nulos, categorias e vetores
- Treinar e avaliar um modelo de classificação
- Escolher o limiar de decisão de acordo com o negócio
- Ajustar hiperparâmetros com validação cruzada
- Salvar o modelo e usá-lo em batch e em streaming

---
## 1. Teoria

### Quando usar MLlib

| | scikit-learn | Spark MLlib |
|-|-------------|-------------|
| Dados | Cabem na memória de uma máquina | Distribuídos, sem limite prático |
| Entrada | NumPy / Pandas | Spark DataFrame |
| Features | Uma matriz `X` | **Uma coluna do tipo vetor** (`features`) |
| Algoritmos | Muito amplo | Os principais: lineares, árvores, ensembles, clustering, ALS |
| Integração | Notebook, APIs | Mesmo pipeline do ETL e do streaming |

Regra prática: se os dados de treino cabem em uma máquina, scikit-learn costuma ser mais simples. O MLlib vale quando o volume é grande ou quando o modelo precisa rodar dentro de pipelines Spark.

### Os três conceitos centrais

| Conceito | O que é | Método | Exemplos |
|----------|---------|--------|----------|
| **Transformer** | Recebe um DataFrame e devolve outro com colunas novas | `.transform(df)` | `VectorAssembler`, um modelo treinado |
| **Estimator** | Aprende algo dos dados e produz um Transformer | `.fit(df)` | `StringIndexer`, `StandardScaler`, `LogisticRegression` |
| **Pipeline** | Sequência de estágios; ele próprio é um Estimator | `.fit(df)` gera um `PipelineModel` | |

```
                       Pipeline.fit(treino)
  ┌──────────────┬──────────────┬───────────────┬──────────────┬──────────────────┐
  │ Imputer      │ StringIndexer│ OneHotEncoder │ Assembler    │ LogisticRegression│  Estimators
  └──────┬───────┴──────┬───────┴───────┬───────┴──────┬───────┴─────────┬────────┘
         ▼              ▼               ▼              ▼                 ▼
  ┌──────────────┬──────────────┬───────────────┬──────────────┬──────────────────┐
  │ ImputerModel │ IndexerModel │ EncoderModel  │ Assembler    │ LRModel          │  PipelineModel
  └──────────────┴──────────────┴───────────────┴──────────────┴──────────────────┘
                       PipelineModel.transform(novos_dados) -> previsões
```

O ponto mais importante: **tudo que aprende dos dados (médias, categorias, escalas) é ajustado só no treino**. Se a média usada para preencher nulos for calculada com os dados de teste, a avaliação fica otimista (*data leakage*). O Pipeline garante isso automaticamente.

---
## 2. Setup e dados

In [ ]:
# !pip install -q pyspark==4.0.1 numpy

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array

spark = (
    SparkSession.builder
    .appName("Aula 7 - MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Geramos 200 mil clientes sintéticos. A probabilidade de churn segue uma regra conhecida (mais chamados de suporte e plano básico aumentam o risco; tempo de casa, pedidos e boa avaliação diminuem). Assim conseguimos verificar se o modelo aprendeu o padrão certo.

Também inserimos **valores nulos** em `last_rating` e `monthly_spend`, como acontece em dados reais.

In [ ]:
N = 200_000
REGIONS = ["North", "Northeast", "Midwest", "Southeast", "South"]

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

base = (
    spark.range(N).withColumnRenamed("id", "customer_id")
    .withColumn("tenure_months",   (F.rand(1) * 71 + 1).cast("int"))
    .withColumn("plan",            pick(["Basic", "Standard", "Premium"], 2))
    .withColumn("region",          pick(REGIONS, 3))
    .withColumn("monthly_spend",   F.round(
        F.when(F.col("plan") == "Basic", 40).when(F.col("plan") == "Standard", 120).otherwise(300)
        * (0.5 + F.rand(4)), 2))
    .withColumn("n_orders_90d",    (F.rand(5) * 15).cast("int"))
    .withColumn("support_tickets", (F.pow(F.rand(6), 2) * 9).cast("int"))
    .withColumn("has_discount",    F.rand(7) < 0.3)
    .withColumn("last_rating",     (F.rand(8) * 5 + 1).cast("int"))
)

logit = (
    0.8
    - 0.045 * F.col("tenure_months")
    + 0.45  * F.col("support_tickets")
    - 0.10  * F.col("n_orders_90d")
    - 0.45  * (F.col("last_rating") - 3)
    + F.when(F.col("plan") == "Basic", 0.5).when(F.col("plan") == "Premium", -0.6).otherwise(0.0)
    - 0.5   * F.col("has_discount").cast("int")
    + 0.6   * F.randn(9)
)

customers = (
    base
    .withColumn("churn", (F.rand(10) < 1 / (1 + F.exp(-logit))).cast("int"))
    .withColumn("last_rating",   F.when(F.rand(11) < 0.06, None).otherwise(F.col("last_rating")))
    .withColumn("monthly_spend", F.when(F.rand(12) < 0.03, None).otherwise(F.col("monthly_spend")))
)

customers.write.mode("overwrite").parquet("/tmp/aula7/customers")
customers = spark.read.parquet("/tmp/aula7/customers")
customers.show(5)

### 2.1 Exploração rápida

In [ ]:
customers.select(F.round(F.avg("churn") * 100, 1).alias("churn_pct")).show()

customers.groupBy("plan").agg(
    F.count("*").alias("clientes"),
    F.round(F.avg("churn") * 100, 1).alias("churn_pct")
).orderBy("churn_pct").show()

customers.groupBy("support_tickets").agg(
    F.round(F.avg("churn") * 100, 1).alias("churn_pct")
).orderBy("support_tickets").show()

print("Nulos:")
customers.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                  for c in ["last_rating", "monthly_spend"]]).show()

---
## 3. Treino e teste

A divisão vem **antes** de qualquer preparação de features. `randomSplit` com `seed` torna a divisão reproduzível.

In [ ]:
train, test = customers.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"Treino: {train.count():,} | Teste: {test.count():,}")

---
## 4. Preparação de features

Os algoritmos do MLlib esperam **uma única coluna de vetor numérico**. Cada tipo de coluna precisa de um tratamento:

| Coluna | Problema | Estágio |
|--------|----------|---------|
| `last_rating`, `monthly_spend` | Nulos | `Imputer` (mediana) |
| `plan`, `region` | Texto | `StringIndexer` (texto -> índice) + `OneHotEncoder` (índice -> vetor binário) |
| Todas | Várias colunas | `VectorAssembler` (junta em um vetor) |
| Numéricas | Escalas diferentes | `StandardScaler` (importante para modelos lineares) |

Vamos ver o efeito de cada estágio isoladamente em algumas linhas.

In [ ]:
sample = train.limit(5)

imputer = Imputer(strategy="median",
                  inputCols=["last_rating", "monthly_spend"],
                  outputCols=["last_rating_imp", "monthly_spend_imp"])
step1 = imputer.fit(train).transform(sample)
step1.select("last_rating", "last_rating_imp", "monthly_spend", "monthly_spend_imp").show()

In [ ]:
indexer = StringIndexer(inputCols=["plan", "region"], outputCols=["plan_idx", "region_idx"],
                        handleInvalid="keep")          # categoria nova no futuro -> índice extra
encoder = OneHotEncoder(inputCols=["plan_idx", "region_idx"], outputCols=["plan_vec", "region_vec"])

idx_model = indexer.fit(train)
print("Ordem das categorias (da mais frequente para a menos):", idx_model.labelsArray)

step2 = encoder.fit(idx_model.transform(train)).transform(idx_model.transform(sample))
step2.select("plan", "plan_idx", "plan_vec", "region", "region_vec").show(truncate=False)

`(3,[1],[1.0])` é um **vetor esparso**: tamanho 3, valor 1.0 na posição 1, zeros no resto.

Por que tamanho 3 se há 3 planos? Com `handleInvalid="keep"`, o `StringIndexer` reserva um índice extra para categorias desconhecidas (4 no total). O `OneHotEncoder` descarta a última posição (`dropLast=True`), que é justamente a de "desconhecido". Uma categoria nova vira um vetor só de zeros.

In [ ]:
NUMERIC = ["tenure_months", "monthly_spend_imp", "n_orders_90d", "support_tickets", "last_rating_imp"]

assembler = VectorAssembler(inputCols=NUMERIC + ["has_discount", "plan_vec", "region_vec"],
                            outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False)

prep = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler]).fit(train)
prep.transform(sample).select("customer_id", "features").show(3, truncate=False)

---
## 5. Primeiro modelo: regressão logística

O pipeline completo é a preparação mais o modelo. Um único `fit` ajusta todos os estágios, na ordem, usando só o treino.

In [ ]:
lr = LogisticRegression(featuresCol="features", labelCol="churn", maxIter=50)
lr_pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler, lr])

t0 = time.perf_counter()
lr_model = lr_pipeline.fit(train)
print(f"Treino: {time.perf_counter() - t0:.1f} s")

pred = lr_model.transform(test)
pred.select("customer_id", "churn", "probability", "prediction").show(5, truncate=False)

Três colunas novas:
- `rawPrediction`: pontuação bruta do modelo.
- `probability`: vetor `[P(classe 0), P(classe 1)]`.
- `prediction`: classe final, usando o limiar padrão de 0.5.

### 5.1 Avaliação

| Métrica | Pergunta | Quando importa |
|---------|----------|----------------|
| **AUC-ROC** | O modelo ordena bem os clientes por risco? | Comparar modelos, independente do limiar |
| **AUC-PR** | Idem, focando na classe positiva | Classes desbalanceadas |
| **Precision** | Dos que marquei como churn, quantos eram? | Custo de abordar um cliente sem necessidade |
| **Recall** | Dos que eram churn, quantos encontrei? | Custo de perder um cliente |
| **Accuracy** | Quantos acertei no total? | Enganosa com classes desbalanceadas |

In [ ]:
def evaluate(predictions, name="modelo"):
    auc_roc = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC").evaluate(predictions)
    auc_pr  = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderPR").evaluate(predictions)
    mc = lambda m, **kw: MulticlassClassificationEvaluator(labelCol="churn", metricName=m, **kw).evaluate(predictions)
    print(f"{name:<22} AUC-ROC={auc_roc:.3f}  AUC-PR={auc_pr:.3f}  "
          f"accuracy={mc('accuracy'):.3f}  precision={mc('precisionByLabel', metricLabel=1.0):.3f}  "
          f"recall={mc('recallByLabel', metricLabel=1.0):.3f}")

evaluate(pred, "Regressão logística")

print("\nMatriz de confusão:")
pred.groupBy("churn").pivot("prediction", [0.0, 1.0]).count().orderBy("churn").show()

### 5.2 O que o modelo aprendeu

Os coeficientes indicam a direção de cada feature: positivo aumenta o risco de churn. Como as features foram padronizadas, as magnitudes são comparáveis.

O `VectorAssembler` guarda nos **metadados** da coluna o nome de cada posição do vetor.

In [ ]:
def feature_names(df, col="features_raw"):
    attrs = df.schema[col].metadata["ml_attr"]["attrs"]
    return [a["name"] for a in sorted((a for group in attrs.values() for a in group), key=lambda a: a["idx"])]

names = feature_names(pred)
coefs = lr_model.stages[-1].coefficients.toArray()

(spark.createDataFrame([(n, float(c)) for n, c in zip(names, coefs)], ["feature", "coef"])
 .orderBy(F.desc(F.abs("coef")))
 .show(truncate=False))

O modelo recuperou a regra usada para gerar os dados: `support_tickets` e o plano básico aumentam o risco; `tenure_months`, `last_rating`, `n_orders_90d` e desconto diminuem; a região quase não importa.

### 5.3 Escolhendo o limiar de decisão

O limiar de 0.5 raramente é o ideal. A escolha é uma **decisão de negócio**: se ligar para um cliente custa pouco e perder um cliente custa muito, aceitamos mais falsos positivos para aumentar o recall.

In [ ]:
scored = pred.select("churn", vector_to_array("probability")[1].alias("p_churn")).cache()

rows = []
for th in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    r = scored.agg(
        F.sum(((F.col("p_churn") >= th) & (F.col("churn") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("p_churn") >= th) & (F.col("churn") == 0)).cast("int")).alias("fp"),
        F.sum(((F.col("p_churn") <  th) & (F.col("churn") == 1)).cast("int")).alias("fn"),
    ).first()
    precision = r.tp / (r.tp + r.fp) if r.tp + r.fp else 0.0
    recall    = r.tp / (r.tp + r.fn)
    rows.append((th, r.tp + r.fp, round(precision, 3), round(recall, 3)))

spark.createDataFrame(rows, ["limiar", "clientes_acionados", "precision", "recall"]).show()

Para aplicar outro limiar no próprio modelo: `lr_model.stages[-1].setThreshold(0.3)`.

---
## 6. Segundo modelo: Random Forest

Modelos de árvore capturam interações e não-linearidades, e não precisam de escala. Reaproveitamos os mesmos estágios de preparação.

| | Regressão logística | Random Forest | Gradient-Boosted Trees |
|-|--------------------|---------------|------------------------|
| Relações | Lineares | Não-lineares | Não-lineares |
| Interpretação | Coeficientes | Importância de features | Importância de features |
| Treino | Rápido | Paralelo (árvores independentes) | Sequencial (mais lento) |
| Risco de overfitting | Baixo | Médio | Maior, exige tuning |

In [ ]:
rf = RandomForestClassifier(featuresCol="features_raw", labelCol="churn",
                            numTrees=60, maxDepth=8, seed=42)
rf_pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, rf])

t0 = time.perf_counter()
rf_model = rf_pipeline.fit(train)
print(f"Treino: {time.perf_counter() - t0:.1f} s")

rf_pred = rf_model.transform(test)
evaluate(pred,    "Regressão logística")
evaluate(rf_pred, "Random Forest")

importances = rf_model.stages[-1].featureImportances.toArray()
(spark.createDataFrame([(n, float(v)) for n, v in zip(feature_names(rf_pred), importances)],
                       ["feature", "importance"])
 .orderBy(F.desc("importance")).show(6, truncate=False))

Aqui a regressão logística empata ou supera a Random Forest. Não é surpresa: os dados foram gerados por uma regra linear. **O modelo mais complexo não é automaticamente o melhor**; sempre compare com uma base simples.

---
## 9. Exercícios

In [ ]:
# ── Exercício 1 ─────────────────────────────────────────────────────────────
# Crie a feature derivada spend_per_order = monthly_spend_imp / (n_orders_90d + 1)
# e inclua-a no pipeline da regressão logística. O AUC melhora? Por quê?
# Dica: use SQLTransformer para criar a coluna DENTRO do pipeline:
#   from pyspark.ml.feature import SQLTransformer
#   SQLTransformer(statement="SELECT *, ... AS spend_per_order FROM __THIS__")

# Sua solução:

In [ ]:
# ── Exercício 2 ─────────────────────────────────────────────────────────────
# Treine um GBTClassifier (maxIter=30, maxDepth=4) com a mesma preparação da Random Forest.
# Compare AUC-ROC e tempo de treino com os outros dois modelos.

# Sua solução:

In [ ]:
# ── Exercício 3 ─────────────────────────────────────────────────────────────
# O time de retenção quer encontrar pelo menos 80% dos clientes que vão cancelar.
# Encontre o MAIOR limiar que garante recall >= 0.80 no teste e informe a precision
# e quantos clientes seriam acionados.

# Sua solução:

In [ ]:
# ── Exercício 4 ─────────────────────────────────────────────────────────────
# Regressão: preveja monthly_spend a partir das outras colunas (sem usar churn).
# Use LinearRegression, descarte linhas com monthly_spend nulo e avalie com
# RegressionEvaluator (RMSE e R2). Qual feature explica a maior parte do gasto?

# Sua solução:

### Soluções

In [ ]:
# ── Solução 1 ───────────────────────────────────────────────────────────────
from pyspark.ml.feature import SQLTransformer

ratio = SQLTransformer(statement=
    "SELECT *, monthly_spend_imp / (n_orders_90d + 1) AS spend_per_order FROM __THIS__")
assembler_v2 = VectorAssembler(inputCols=NUMERIC + ["spend_per_order", "has_discount", "plan_vec", "region_vec"],
                               outputCol="features_raw")

lr_v2 = Pipeline(stages=[imputer, ratio, indexer, encoder, assembler_v2, scaler, lr]).fit(train)
evaluate(pred, "LR original")
evaluate(lr_v2.transform(test), "LR + spend_per_order")
# Não melhora: o churn foi gerado sem nenhuma relação com essa razão.
# Engenharia de features só ajuda quando captura um padrão real do problema.

In [ ]:
# ── Solução 2 ───────────────────────────────────────────────────────────────
gbt = GBTClassifier(featuresCol="features_raw", labelCol="churn", maxIter=30, maxDepth=4, seed=42)

t0 = time.perf_counter()
gbt_model = Pipeline(stages=[imputer, indexer, encoder, assembler, gbt]).fit(train)
print(f"Treino GBT: {time.perf_counter() - t0:.1f} s")

evaluate(pred,                        "Regressão logística")
evaluate(rf_pred,                     "Random Forest")
evaluate(gbt_model.transform(test),   "GBT")

In [ ]:
# ── Solução 3 ───────────────────────────────────────────────────────────────
total_churn = scored.filter("churn = 1").count()

best_th = None
for th in [x / 100 for x in range(5, 96)]:
    tp = scored.filter((F.col("p_churn") >= th) & (F.col("churn") == 1)).count()
    if tp / total_churn >= 0.80:
        best_th = th
    else:
        break

flagged = scored.filter(F.col("p_churn") >= best_th)
tp = flagged.filter("churn = 1").count()
print(f"Limiar = {best_th:.2f} | recall = {tp / total_churn:.3f} | "
      f"precision = {tp / flagged.count():.3f} | clientes acionados = {flagged.count():,}")

In [ ]:
# ── Solução 4 ───────────────────────────────────────────────────────────────
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

train_r = train.filter(F.col("monthly_spend").isNotNull())
test_r  = test.filter(F.col("monthly_spend").isNotNull())

imp_r = Imputer(strategy="median", inputCols=["last_rating"], outputCols=["last_rating_imp"])
asm_r = VectorAssembler(
    inputCols=["tenure_months", "n_orders_90d", "support_tickets", "last_rating_imp",
               "has_discount", "plan_vec", "region_vec"],
    outputCol="features_raw")
reg = LinearRegression(featuresCol="features_raw", labelCol="monthly_spend")

reg_model = Pipeline(stages=[imp_r, indexer, encoder, asm_r, reg]).fit(train_r)
reg_pred = reg_model.transform(test_r)

for m in ["rmse", "r2"]:
    print(m.upper(), "=", round(RegressionEvaluator(labelCol="monthly_spend", metricName=m).evaluate(reg_pred), 3))

(spark.createDataFrame(list(zip(feature_names(reg_pred), map(float, reg_model.stages[-1].coefficients))),
                       ["feature", "coef"])
 .orderBy(F.desc(F.abs("coef"))).show(4, truncate=False))
# O plano domina: o gasto foi gerado como (valor do plano) x (fator aleatório).

In [ ]:
spark.stop()